# SupportIQ AI: 01 Explore the data and check the labels
Run this first. It loads the Kaggle *Customer Support Ticket Dataset*, explores it, and tests whether the labels can be predicted from the ticket text at all. Use **Runtime > Run all**.

In [ ]:
!pip -q install kagglehub scikit-learn==1.6.1 pandas matplotlib

## 1. Load the dataset

In [ ]:
import glob, os
try:
    import kagglehub
    path = kagglehub.dataset_download("suraj520/customer-support-ticket-dataset")
    CSV = glob.glob(os.path.join(path, "*.csv"))[0]
except Exception as e:
    print("Kaggle download failed:", e, "\nUpload customer_support_tickets.csv manually.")
    from google.colab import files
    up = files.upload()
    CSV = list(up)[0]
print("Using:", CSV)

In [ ]:
import pandas as pd
df = pd.read_csv(CSV)
print(df.shape)
df.head(3).T

## 2. Data quality

In [ ]:
print(df.isna().sum()[lambda s: s > 0])
print("duplicate rows:", df.duplicated().sum())
print("unique descriptions:", df["Ticket Description"].nunique(), "of", len(df))
print("descriptions containing a template placeholder like {product_purchased}:",
      df["Ticket Description"].str.contains(r"\{[^}]*\}", regex=True).sum())

## 3. Label distributions
These are the targets we will predict: **Ticket Type** (category), **Ticket Priority**, **Ticket Subject** (issue topic).

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, col in zip(axes, ["Ticket Type", "Ticket Priority", "Ticket Subject"]):
    df[col].value_counts().sort_values().plot.barh(ax=ax, title=col)
plt.tight_layout(); plt.show()

## 4. Example tickets

In [ ]:
pd.set_option("display.max_colwidth", 220)
df[["Ticket Type", "Ticket Priority", "Ticket Subject", "Ticket Description"]].sample(6, random_state=1)

## 5. Can the labels be predicted from the text?
We compare a TF-IDF + Logistic Regression model against random guessing using 5-fold cross-validation, **grouped by identical text** so duplicates cannot leak between folds.
If a score is close to random guessing, the label does not depend on the text. That is a property of the dataset, not a bug in your code, and the app reports it honestly.

In [ ]:
import re, numpy as np
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline

clean = lambda t: re.sub(r"\s+", " ", re.sub(r"\{[^}]*\}", " ", str(t))).strip().lower()
d = df.dropna(subset=["Ticket Subject", "Ticket Description"]).copy()
d["text"] = [clean(f"{s}. {x}") for s, x in zip(d["Ticket Subject"], d["Ticket Description"])]
groups = d["text"].factorize()[0]
cv = GroupKFold(n_splits=5)

rows = []
for col in ["Ticket Type", "Ticket Priority", "Ticket Subject"]:
    m = make_pipeline(TfidfVectorizer(min_df=2, sublinear_tf=True), LogisticRegression(max_iter=1000, class_weight="balanced"))
    s = cross_val_score(m, d["text"], d[col], cv=cv, groups=groups, scoring="f1_macro").mean()
    b = cross_val_score(DummyClassifier(strategy="stratified", random_state=0), d["text"], d[col], cv=cv, groups=groups, scoring="f1_macro").mean()
    rows.append((col, round(s, 3), round(b, 3), "learns signal" if s > b + 0.05 else "near chance"))
pd.DataFrame(rows, columns=["target", "model macro-F1", "random macro-F1", "verdict"])

### How to read this
* **learns signal**: the model is meaningfully better than guessing. Use it as is.
* **near chance**: the dataset's labels are not tied to the text. Keep training (notebook 02) so the app still works, but present these metrics honestly in your README. The app's urgency rules and "needs review" flag exist for exactly this case.

Next: open **02_train_and_export.ipynb**.